# Optional exercise: forecasting the monthly number of BTI decisions

*Session 12, optional. Author: course team, licence CC-BY-4.0.*

The main practice of this session uses the Airbnb review series ([workbook 10](10-case-study-airbnb-review-forecast.ipynb)). This notebook applies the same workflow to the running case study of the course, the EBTI decisions: a flatter series with a weak season and a different kind of break (the United Kingdom stops issuing decisions after Brexit).

**Question.** How many Binding Tariff Information decisions will the customs authorities of the EU member states issue per month over the next twelve months, and how certain is that forecast? A customs administration could use such a forecast to plan the staff that answers BTI applications.

- **Part 1 (block 1):** build the monthly series from `monthly_counts.parquet`, describe it and compute the baselines.
- **Part 2 (block 2):** fit exponential smoothing with a prediction interval and compare it with the baselines.
- **Part 3 (block 3):** add a lag-feature model, backtest all models with a rolling origin, and recommend one with its prediction interval.
- **Part 4 (optional):** the same comparison for the largest chapters, many small series instead of one.

Theory: [01](../theory/01-time-series-and-baselines.md), [02](../theory/02-exponential-smoothing-and-arima.md), [03](../theory/03-lag-features-and-backtesting.md).

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore")     # statsmodels convergence notes
pd.set_option("display.width", 120)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"


# Part 1: the series and its baselines

## 1.1 From the count table to a monthly series

`monthly_counts` has one row per month (of the start of validity), issuing country and chapter.

In [ ]:
counts = pd.read_parquet(DATA / "monthly_counts.parquet")
print(counts.head(3).to_string())
by_country = counts.pivot_table(index=counts["month"].dt.year, columns="issuing_country",
                                values="n_decisions", aggfunc="sum", fill_value=0)
by_country.loc[2004:2026, ["DE", "FR", "GB", "NL", "PL"]]

**Questions.** (a) What happens to the United Kingdom (`GB`) after 2020? (b) Look at the months after September 2026 (`counts[counts["month"] > "2026-09-01"]`). Why must they be dropped?

In [ ]:
eu = counts[counts["issuing_country"] != "GB"]
y = eu.groupby("month")["n_decisions"].sum()["2010":"2026-09"].rename("decisions")
y.index.freq = "MS"
log_y = np.log(y)
ax = y.plot(figsize=(10, 3.5), title="BTI decisions per month, EU member states without GB")
ax.set_ylabel("decisions")
len(y)

## 1.2 Trend, seasonality, autocorrelation

In [ ]:
stl = STL(log_y, period=12, robust=True).fit()
stl.plot()
plt.tight_layout()
seasonal_factor = np.exp(stl.seasonal).groupby(y.index.month).mean().round(2)
seasonal_factor.rename("seasonal factor").to_frame().T

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
plot_acf(y, lags=24, ax=a1, title="ACF of the counts")
plot_acf(log_y.diff().dropna(), lags=24, ax=a2, title="ACF of monthly growth")
plt.tight_layout()

**Question.** Which months are high, which low? Is the yearly pattern strong compared with the month-to-month noise?

## 1.3 Baselines on the hold-out year (October 2025 to September 2026)

In [ ]:
train, test = y[:"2025-09"], y["2025-10":]
H = len(test)


def mae(actual, forecast):
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))))


def mase(actual, forecast, y_tr, m=12):
    history = np.asarray(y_tr)
    return mae(actual, forecast) / np.mean(np.abs(history[m:] - history[:-m]))


def baselines(y_tr, index):
    return {
        "naive": pd.Series(y_tr.iloc[-1], index=index),
        "seasonal naive": pd.Series(y_tr.iloc[-12:].to_numpy(), index=index),
        "moving average (3)": pd.Series(y_tr.iloc[-3:].mean(), index=index),
    }


base = baselines(train, test.index)
ax = y["2021":].plot(figsize=(10, 3.5), color="black", label="actual")
for name, fc in base.items():
    fc.plot(ax=ax, ls="--", label=name)
ax.legend()
pd.DataFrame({name: {"MAE": mae(test, fc), "MASE": mase(test, fc, train)} for name, fc in base.items()}).T.round(2)

# Part 2: exponential smoothing with a prediction interval

We fit an ETS model on the log counts: additive errors, a damped additive trend and additive seasonality (on the log scale, i.e. multiplicative on the original scale).

In [ ]:
def fit_ets(y_tr):
    return ETSModel(np.log(y_tr), error="add", trend="add", damped_trend=True,
                    seasonal="add", seasonal_periods=12).fit(disp=False)


ets = fit_ets(train)
print(dict(zip(ets.param_names[:4], ets.params[:4].round(3))))
frame = np.exp(ets.get_prediction(start=test.index[0], end=test.index[-1]).summary_frame(alpha=0.05))

ax = y["2021":].plot(figsize=(10, 3.5), color="black", label="actual")
ax.fill_between(frame.index, frame["pi_lower"], frame["pi_upper"], alpha=0.2, label="95 % interval")
frame["mean"].plot(ax=ax, ls="--", label="ETS")
ax.legend(loc="upper left")
inside = ((test >= frame["pi_lower"]) & (test <= frame["pi_upper"])).sum()
print(f"ETS MAE {mae(test, frame['mean']):.0f}, MASE {mase(test, frame['mean'], train):.2f}, {inside} of {H} months inside the interval")

**Exercise 2.1.** Fit the same model without the trend (`trend=None`) and without the log transform. Compare MAE and the width of the December interval.

In [ ]:
# Exercise 2.1: your code here


# Part 3: lag features, backtest and recommendation

## 3.1 A lag-feature model (direct, 12 months ahead)

Only lags of 12 or more months are used, so that every feature is known when the forecast is made.

In [ ]:
table = pd.DataFrame({"target": log_y})
for lag in (12, 13, 24):
    table[f"lag{lag}"] = log_y.shift(lag)
table["month"] = table.index.month
table = table.dropna()


def fit_lag_model(y_tr):
    rows = table.loc[:y_tr.index[-1]]
    return HistGradientBoostingRegressor(min_samples_leaf=5, random_state=0).fit(
        rows.drop(columns="target"), rows["target"])


lag_model = fit_lag_model(train)
lag_fc = pd.Series(np.exp(lag_model.predict(table.loc[test.index].drop(columns="target"))), index=test.index)
print(f"lag model MAE {mae(test, lag_fc):.0f}, MASE {mase(test, lag_fc, train):.2f}")

## 3.2 Rolling-origin backtest

Five folds: the origin moves forward twelve months at a time; the test periods run from October to September, the first from October 2021 to September 2022.

In [ ]:
rows = []
for train_idx, test_idx in TimeSeriesSplit(n_splits=5, test_size=12).split(y):
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    fcs = baselines(y_tr, y_te.index)
    pred = np.exp(fit_ets(y_tr).get_prediction(start=y_te.index[0], end=y_te.index[-1]).summary_frame(alpha=0.05))
    fcs["ETS"] = pred["mean"]
    fcs["lag model"] = np.exp(fit_lag_model(y_tr).predict(table.loc[y_te.index].drop(columns="target")))
    coverage = ((y_te >= pred["pi_lower"]) & (y_te <= pred["pi_upper"])).mean()
    for name, fc in fcs.items():
        rows.append({"test from": y_te.index[0].strftime("%Y-%m"), "model": name, "MAE": mae(y_te, fc),
                     "MASE": mase(y_te, fc, y_tr), "ETS coverage": coverage})
backtest = pd.DataFrame(rows)
backtest.pivot(index="test from", columns="model", values="MASE").round(2)

In [ ]:
summary = backtest.groupby("model").agg(MAE=("MAE", "mean"), MASE=("MASE", "mean"),
                                        MASE_worst=("MASE", "max"))
summary["folds best"] = (backtest.loc[backtest.groupby("test from")["MASE"].idxmin(), "model"]
                         .value_counts().reindex(summary.index).fillna(0).astype(int))
print("ETS interval coverage per fold:", backtest.groupby("test from")["ETS coverage"].first().round(2).to_list())
summary.sort_values("MASE").round(2)

## 3.3 Recommendation

Write your recommendation for the head of a customs administration (4–6 sentences): which model, its forecast for the next twelve months with the 95 % interval, how accurate it was in the backtest compared with the seasonal naive forecast, and what could make the forecast wrong (think of Brexit, the pandemic dip in 2020–2021 and the HS revision of 2022).

The cell below refits the ETS model on all data up to September 2026 and produces the forecast for October 2026 to September 2027.

In [ ]:
final = fit_ets(y)
future = pd.date_range("2026-10-01", periods=12, freq="MS")
outlook = np.exp(final.get_prediction(start=future[0], end=future[-1]).summary_frame(alpha=0.05))
ax = y["2021":].plot(figsize=(10, 3.5), color="black", label="actual")
ax.fill_between(outlook.index, outlook["pi_lower"], outlook["pi_upper"], alpha=0.2, label="95 % interval")
outlook["mean"].plot(ax=ax, ls="--", label="ETS forecast")
ax.legend(loc="upper left")
print(f"expected total October 2026 to September 2027: {outlook['mean'].sum():.0f}")
outlook[["mean", "pi_lower", "pi_upper"]].round(0)

*Your recommendation:* ...

# Part 4 (optional): many small series

A customs office plans per product area, not only in total. We repeat the last backtest fold for the five largest chapters: is ETS still better than seasonal naive when the series are smaller and noisier?

In [ ]:
top = eu.groupby("chapter")["n_decisions"].sum().nlargest(5).index
months = pd.date_range("2010-01-01", "2026-09-01", freq="MS")
rows = []
for chapter in top:
    s = (eu[eu["chapter"] == chapter].groupby("month")["n_decisions"].sum()
         .reindex(months, fill_value=0).astype(float))          # keep empty months as 0
    s.index.freq = "MS"
    s_tr, s_te = s[:"2025-09"], s["2025-10":]
    ets_c = np.exp(ETSModel(np.log1p(s_tr), error="add", trend="add", damped_trend=True,
                            seasonal="add", seasonal_periods=12).fit(disp=False).forecast(12)) - 1
    rows.append({"chapter": chapter, "mean per month": round(s_te.mean()),
                 "MASE ETS": mase(s_te, ets_c, s_tr),
                 "MASE seasonal naive": mase(s_te, s_tr.iloc[-12:].to_numpy(), s_tr)})
pd.DataFrame(rows).round(2)

## Exercises

1. Add a sliding-window variant of the backtest (`TimeSeriesSplit(..., max_train_size=60)`). Do the rankings change?
2. Build the series *including* GB and run the backtest again. Which models suffer from the break after 2020, and why?
3. Plot the chapter-63 series (other made-up textile articles, including face masks) for 2018–2022. Is there a visible COVID effect in 2020? Count the training decisions with the keyword `MASK` per year to explain what you see.
4. Optional: repeat the backtest with StatsForecast (`AutoETS`, `SeasonalNaive`, `cross_validation`), as in workbook 08, and compare the results.

In [ ]:
# Exercise 1: your code here
